# Debugging Bias in Hiring Algorithms: A Comparative Study of Fairness-Aware Mitigation Strategies

**Authors:** Bibha Ray, Rajad Shakya  
**Institution:** Department of Electronics and Computer Engineering, Thapathali Campus, IOE, Tribhuvan University, Kathmandu, Nepal

---

### Mid-term Update Implementation
This notebook covers the machine learning pipeline, data preprocessing, and evaluation of two distinct bias mitigation strategies:
1. **Pre-processing:** Reweighing
2. **Post-processing:** Threshold Optimization

**Dataset:** [70k+ Job Applicants Data (Human Resource)](https://www.kaggle.com/datasets/ayushtankha/70k-job-applicants-data-human-resource)

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import warnings

warnings.filterwarnings('ignore')

In [9]:
# 1. Load data
# Ensure the downloaded csv file is in the same directory
file_path = 'sample_data/job_applicants_data.csv'

try:
    df = pd.read_csv(file_path)
    print("Dataset loaded successfully. Shape:", df.shape)
except FileNotFoundError:
    print("File not found. Please make sure the Kaggle dataset is downloaded.")
    print("Generating dummy dataset with matching schema to demonstrate the pipeline...")

    np.random.seed(42)
    n_samples = 5000
    df = pd.DataFrame({
        'Gender': np.random.choice(['Man', 'Woman', 'NonBinary'], n_samples, p=[0.6, 0.35, 0.05]),
        'Age': np.random.choice(['<35', '>35'], n_samples),
        'Accessibility': np.random.choice(['Yes', 'No'], n_samples, p=[0.05, 0.95]),
        'MentalHealth': np.random.choice(['Yes', 'No'], n_samples, p=[0.1, 0.9]),
        'YearsCode': np.random.randint(0, 25, n_samples),
        'YearsCodePro': np.random.randint(0, 20, n_samples),
        'PreviousSalary': np.random.normal(80000, 25000, n_samples),
        'EdLevel': np.random.choice(['Undergraduate', 'Master', 'PhD', 'Other', 'NoHigherEd'], n_samples),
        'ComputerSkills': np.random.randint(1, 15, n_samples),
        'Employed': np.random.choice([0, 1], n_samples, p=[0.7, 0.3])
    })

df.head()

Dataset loaded successfully. Shape: (73462, 15)


,Unnamed: 0,Age,Accessibility,EdLevel,Employment,Gender,MentalHealth,MainBranch,YearsCode,YearsCodePro,Country,PreviousSalary,HaveWorkedWith,ComputerSkills,Employed
0,0,<35,No,Master,1,Man,No,Dev,7,4,Sweden,51552.0,C++;Python;Git;PostgreSQL,4,0
1,1,<35,No,Undergraduate,1,Man,No,Dev,12,5,Spain,46482.0,Bash/Shell;HTML/CSS;JavaScript;Node.js;SQL;Typ...,12,1
2,2,<35,No,Master,1,Man,No,Dev,15,6,Germany,77290.0,C;C++;Java;Perl;Ruby;Git;Ruby on Rails,7,0
3,3,<35,No,Undergraduate,1,Man,No,Dev,9,6,Canada,46135.0,Bash/Shell;HTML/CSS;JavaScript;PHP;Ruby;SQL;Gi...,13,0
4,4,>35,No,PhD,0,Man,No,NotDev,40,30,Singapore,160932.0,C++;Python,2,0


In [10]:
# 2. Data Preprocessing

# Drop Unnamed index column if it exists
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

# Map education as an ordinal feature
edu_mapping = {'Undergraduate': 1, 'Master': 2, 'PhD': 3, 'Other': 0, 'NoHigherEd': 0}
if 'EdLevel' in df.columns:
    df['Education_encoded'] = df['EdLevel'].map(edu_mapping).fillna(0)

# One-hot encoding categorical attributes (Demographic & Social)
cat_cols = ['Gender', 'Age', 'Accessibility', 'MentalHealth']
cat_cols_present = [c for c in cat_cols if c in df.columns]

df_encoded = pd.get_dummies(df, columns=cat_cols_present, drop_first=False)

# Standardizing numeric features
num_cols = ['YearsCode', 'YearsCodePro', 'PreviousSalary', 'ComputerSkills']
num_cols_present = [c for c in num_cols if c in df_encoded.columns]

scaler = StandardScaler()
if num_cols_present:
    df_encoded[num_cols_present] = scaler.fit_transform(df_encoded[num_cols_present])

# Isolate target and feature space
target = 'Employed'

# Define useful features explicitly, avoiding arbitrary string columns
encoded_cat_features = [c for c in df_encoded.columns if any(c.startswith(orig_col + '_') for orig_col in cat_cols_present)]
features = num_cols_present + encoded_cat_features + ['Education_encoded']
features = [f for f in features if f in df_encoded.columns]

X = df_encoded[features]
y = df_encoded[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print("Training set shape:", X_train.shape)

Training set shape: (51423, 14)


In [11]:
# 3. Evaluation Metrics Definition

def calculate_fairness(y_true, y_pred, mask_unprivileged):
    """
    mask_unprivileged: boolean series where True indicates the disadvantaged group
    """
    acc = accuracy_score(y_true, y_pred)
    mask_privileged = ~mask_unprivileged

    # Disparate Impact Ratio (DIR) threshold target > 0.8
    sr_unpriv = y_pred[mask_unprivileged].mean()
    sr_priv = y_pred[mask_privileged].mean()
    dir_score = sr_unpriv / sr_priv if sr_priv > 0 else 0

    # Equalized Odds (TPR diff) target ~ 0
    try:
        tp_unpriv = ((y_pred[mask_unprivileged] == 1) & (y_true[mask_unprivileged] == 1)).sum()
        actual_p_unpriv = (y_true[mask_unprivileged] == 1).sum()
        tpr_unpriv = tp_unpriv / actual_p_unpriv if actual_p_unpriv > 0 else 0

        tp_priv = ((y_pred[mask_privileged] == 1) & (y_true[mask_privileged] == 1)).sum()
        actual_p_priv = (y_true[mask_privileged] == 1).sum()
        tpr_priv = tp_priv / actual_p_priv if actual_p_priv > 0 else 0

        eod_tpr = abs(tpr_unpriv - tpr_priv)
    except:
        eod_tpr = np.nan

    return {
        'Accuracy': round(acc, 4),
        'DIR': round(dir_score, 4),
        'EOD_TPR_Diff': round(eod_tpr, 4)
    }

In [12]:
# Identify intersectional group based on research gap: Women > 35

try:
    # Identifying exact dummy-generated column names
    col_woman = [c for c in X_train.columns if 'Gender_Woman' in c][0]
    col_age = [c for c in X_train.columns if 'Age_>35' in c][0]

    mask_train_intersectional = (X_train[col_woman] == 1) & (X_train[col_age] == 1)
    mask_test_intersectional = (X_test[col_woman] == 1) & (X_test[col_age] == 1)
    print("Intersectional group defined: Women > 35")
except IndexError:
    print("Could not find exact column names for Gender and Age. Check exact header names.")
    # safe fallback
    mask_train_intersectional = pd.Series([False]*len(X_train), index=X_train.index)
    mask_test_intersectional = pd.Series([False]*len(X_test), index=X_test.index)

Intersectional group defined: Women > 35


In [13]:
# 4. Baseline Model (No Mitigation)

rf_base = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_base.fit(X_train, y_train)

y_pred_base = rf_base.predict(X_test)
metrics_base = calculate_fairness(y_test, y_pred_base, mask_test_intersectional)

print("--- Baseline Model Metrics ---")
print(metrics_base)

--- Baseline Model Metrics ---
{'Accuracy': 0.7833, 'DIR': np.float64(0.6889), 'EOD_TPR_Diff': np.float64(0.1002)}


In [14]:
# 5. Strategy 1: Pre-processing (Reweighing)

df_train = X_train.copy()
df_train['y'] = y_train
df_train['is_unpriv'] = mask_train_intersectional

weights = np.ones(len(df_train))

for group_val in [True, False]:
    for label_val in [0, 1]:
        cond = (df_train['is_unpriv'] == group_val) & (df_train['y'] == label_val)
        if cond.sum() == 0:
            continue

        # formula: W = P(Group)*P(Class) / P(Group and Class)
        p_group = (df_train['is_unpriv'] == group_val).mean()
        p_class = (df_train['y'] == label_val).mean()
        p_group_and_class = cond.mean()

        weight = (p_group * p_class) / p_group_and_class
        weights[cond] = weight

rf_reweigh = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_reweigh.fit(X_train, y_train, sample_weight=weights)

y_pred_rw = rf_reweigh.predict(X_test)
metrics_rw = calculate_fairness(y_test, y_pred_rw, mask_test_intersectional)

print("--- Pre-processing (Reweighing) Metrics ---")
print(metrics_rw)

--- Pre-processing (Reweighing) Metrics ---
{'Accuracy': 0.7827, 'DIR': np.float64(0.7028), 'EOD_TPR_Diff': np.float64(0.0901)}


In [15]:
# 6. Strategy 2: Post-processing (Threshold Optimization)

probas = rf_base.predict_proba(X_test)[:, 1]

# predictions for privileged group stay at default 50% threshold
mask_priv = ~mask_test_intersectional
y_pred_priv = (probas[mask_priv] >= 0.5).astype(int)

# Calculate Equalized Odds Target (TPR of privileged group)
actual_p_priv = (y_test[mask_priv] == 1).sum()
tpr_target = ((y_pred_priv == 1) & (y_test[mask_priv] == 1)).sum() / actual_p_priv if actual_p_priv > 0 else 0

# Search for a customized threshold for unprivileged group
best_t = 0.5
min_diff = float('inf')

for t in np.arange(0.1, 0.9, 0.05):
    y_pred_temp = (probas[mask_test_intersectional] >= t).astype(int)
    actual_p_unpriv = (y_test[mask_test_intersectional] == 1).sum()

    if actual_p_unpriv > 0:
        tpr_temp = ((y_pred_temp == 1) & (y_test[mask_test_intersectional] == 1)).sum() / actual_p_unpriv
        diff = abs(tpr_temp - tpr_target)
        if diff < min_diff:
            min_diff = diff
            best_t = t

print(f"Optimized threshold for unprivileged intersectional group: {best_t:.4f}")

y_pred_post = np.zeros_like(y_pred_base)
y_pred_post[mask_priv] = (probas[mask_priv] >= 0.5).astype(int)
y_pred_post[mask_test_intersectional] = (probas[mask_test_intersectional] >= best_t).astype(int)

metrics_post = calculate_fairness(y_test, y_pred_post, mask_test_intersectional)

print("--- Post-processing (Threshold Optimization) Metrics ---")
print(metrics_post)

Optimized threshold for unprivileged intersectional group: 0.3500
--- Post-processing (Threshold Optimization) Metrics ---
{'Accuracy': 0.7832, 'DIR': np.float64(0.8641), 'EOD_TPR_Diff': np.float64(0.0046)}


In [16]:
# 7. Final Methods Comparison

comparison_df = pd.DataFrame([
    metrics_base,
    metrics_rw,
    metrics_post
], index=['Baseline (No Mitigation)',
          'Reweighing (Pre-processing)',
          'Threshold Optimization (Post-processing)'])

comparison_df

,Accuracy,DIR,EOD_TPR_Diff
Baseline (No Mitigation),0.7833,0.6889,0.1002
Reweighing (Pre-processing),0.7827,0.7028,0.0901
Threshold Optimization (Post-processing),0.7832,0.8641,0.0046
